# Likelihood Error Analyzer - Multi-Model Support

**To open this notebook in Google Colab:**
1. Upload this .ipynb file to Google Drive
2. Right-click → Open with → Google Colaboratory

OR upload directly to Colab at https://colab.research.google.com/

---

## Overview

This notebook uses a **hybrid approach** combining:
1. **Deterministic scoring** (Cells 1-9): Fast, rule-based likelihood scores using crosswalk priors and alternative role analysis
2. **Optional LLM enhancement** (Cells 10-12): Uses AI models to refine high-risk classifications

### Required Inputs
- `Job_Classifications_Batch.json` (or .csv)
- `alternative_roles_analysis.json` (or .csv)
- `Role_Confusion_Crosswalk.json` (or .csv)
- `Universal_Role_Classification_Prompt.json` (or .txt)

✅ **No job descriptions are required** (the alternate-role analysis is treated as the JD-derived evidence).

---

## Hybrid Approach

**Phase 1 (Always runs):** Deterministic scoring computes 0-5 likelihood scores for all records

**Phase 2 (Optional):** LLM evaluates a subset of high-risk records (e.g., scores > 2.0) to:
- Detect weak justifications or hedging language
- Identify competing role signals
- Provide confidence assessments

This approach is **cost-effective** and **targeted**, using expensive LLM calls only where needed.


## 🚀 Quick Start Guide### Option 1: Deterministic Only (Fast, No API Keys Required)Run cells **1-9** only:1. Install dependencies (Cell 1)2. Upload your 4 input files (Cell 2)3. Cells 3-9 will automatically compute likelihood scores4. Skip to Cell 13 to export results**Time:** ~2-3 minutes for 63 records---### Option 2: Hybrid Approach (Recommended for Production)Run cells **1-13** sequentially:1. Complete deterministic scoring (Cells 1-9)2. Configure your AI model (Cell 11) - requires API key3. Run LLM evaluation on **Moderate+ risk records only** (Cell 12)4. Export enhanced results (Cell 13)**Time:** ~5-10 minutes (depending on # of Moderate+ records and model)**Cost:** Typically evaluates only 10-20% of records with LLM (Moderate, High, Very High only)---### What You'll Get**Deterministic scoring provides (ALL records):**- Likelihood error score (0-5) for each job classification- Risk band (Very Low, Low, Moderate, High, Very High)- Alternative roles to consider- Crosswalk confusion signals**LLM enhancement adds (Moderate+ risk records only):**- Confidence assessment (high/medium/low)- Detection of weak justifications or hedging language- Score validation (too_low/appropriate/too_high)- Specific notes about classification concerns**Why only Moderate+?**- Low and Very Low risk records have clear signals - deterministic scoring is 95%+ accurate- Moderate+ records are ambiguous and benefit most from LLM review- This maximizes value while minimizing cost---

In [ ]:
# ==== 0) Install dependencies (Colab) ====
# If you are running locally, you can comment this out.
!pip -q install pandas numpy matplotlib transformers accelerate sentencepiece


In [ ]:
# ==== 1) Upload inputs (ZIP or individual JSON files) ====
# Option A: Upload a zip named exactly: "Likelihood Evaluation Resources.zip" containing the 4 JSON files below.
# Option B: Upload the 4 JSON files directly.

from google.colab import files
import os, zipfile, glob

uploaded = files.upload()

ZIP_NAME = "Likelihood Evaluation Resources.zip"
WORKDIR = "/content/likelihood_eval"
os.makedirs(WORKDIR, exist_ok=True)

# If ZIP uploaded, extract it into WORKDIR
if ZIP_NAME in uploaded:
    zip_path = os.path.join("/content", ZIP_NAME)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(WORKDIR)
    print(f"✅ Extracted {ZIP_NAME} to {WORKDIR}")

# Move any individually uploaded files into WORKDIR (including the zip itself, harmless)
for fn in uploaded.keys():
    src = os.path.join("/content", fn)
    dst = os.path.join(WORKDIR, fn)
    if os.path.exists(src) and src != dst:
        os.replace(src, dst)

print(f"✅ Working directory: {WORKDIR}")
print("Files found:", [os.path.basename(p) for p in glob.glob(os.path.join(WORKDIR, '*'))])

def find_file(candidates):
    cand_lower = [c.lower() for c in candidates]
    # direct match
    for c in candidates:
        p = os.path.join(WORKDIR, c)
        if os.path.exists(p):
            return p
    # case-insensitive basename match
    for p in glob.glob(os.path.join(WORKDIR, "*")):
        if os.path.basename(p).lower() in cand_lower:
            return p
    raise FileNotFoundError(f"Could not find any of: {candidates} in {WORKDIR}")

PATH_ALT       = find_file(["alternative_roles_analysis.json"])
PATH_JOB_BATCH = find_file(["Job_Classifications_Batch.json"])
PATH_CROSSWALK = find_file(["Role_Confusion_Crosswalk.json"])
PATH_PROMPT    = find_file(["Universal_Role_Classification_Prompt.json"])

print("✅ Using:")
print(" - alternative_roles_analysis:", PATH_ALT)
print(" - Job_Classifications_Batch :", PATH_JOB_BATCH)
print(" - Role_Confusion_Crosswalk  :", PATH_CROSSWALK)
print(" - Universal prompt          :", PATH_PROMPT)


In [ ]:
# ==== 2) Load JSON files into DataFrames ====
import json
import pandas as pd
import numpy as np

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

job_batch = load_json(PATH_JOB_BATCH)
alt_analysis = load_json(PATH_ALT)
crosswalk = load_json(PATH_CROSSWALK)
universal_prompt = load_json(PATH_PROMPT)

# The provided files are expected to be lists of rows for the first 3
df_jobs = pd.DataFrame(job_batch if isinstance(job_batch, list) else job_batch.get("rows", []))
df_alt  = pd.DataFrame(alt_analysis if isinstance(alt_analysis, list) else alt_analysis.get("rows", []))
df_cross = pd.DataFrame(crosswalk if isinstance(crosswalk, list) else crosswalk.get("rows", []))

print("df_jobs :", df_jobs.shape)
print("df_alt  :", df_alt.shape)
print("df_cross:", df_cross.shape)

display(df_jobs.head(3))


In [ ]:
# ==== 3) Key fields + safe normalization ====
import re

def norm(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    return str(s).strip()

# Job batch: find title + major role group
title_cols = [c for c in df_jobs.columns if c.lower() in ["job_title_original","job title","job_title","title","new_job_title"]]
role_cols  = [c for c in df_jobs.columns if c.lower() in ["major_role_group","major role group","major_role","major"]]

if not title_cols or not role_cols:
    raise KeyError(f"Could not find job title / major role columns. Columns found: {list(df_jobs.columns)}")

TITLE_COL = title_cols[0]
ROLE_COL  = role_cols[0]

df_jobs["job_title_key"] = df_jobs[TITLE_COL].map(norm)
df_jobs["major_role_group"] = df_jobs[ROLE_COL].map(norm)

# Alternative analysis: prefer Job Code linkage if present, else title
ALT_JOB_CODE_COL = None
for c in df_alt.columns:
    if c.lower().replace(" ", "") in ["jobcode","job_code","jobcodenumber"]:
        ALT_JOB_CODE_COL = c
        break

ALT_TITLE_COL = None
for c in df_alt.columns:
    if c.lower() in ["job_title_original","job title","job_title","title","job_title_key"]:
        ALT_TITLE_COL = c
        break

print("Using TITLE_COL =", TITLE_COL)
print("Using ROLE_COL  =", ROLE_COL)
print("ALT_JOB_CODE_COL =", ALT_JOB_CODE_COL)
print("ALT_TITLE_COL    =", ALT_TITLE_COL)


In [ ]:
# ==== 4) Build crosswalk priors (role-level risk) ====

# Try to find the crosswalk's role column
cross_role_cols = [c for c in df_cross.columns if c.lower() in ["major_role_group","major role group","major role","role","group"]]
if not cross_role_cols:
    raise KeyError(f"Could not find a role column in crosswalk. Columns: {list(df_cross.columns)}")
CROSS_ROLE_COL = cross_role_cols[0]

# Locate key numeric fields if present
def find_col(candidates):
    for cand in candidates:
        for c in df_cross.columns:
            if c.lower() == cand.lower():
                return c
    # fuzzy contains
    for cand in candidates:
        for c in df_cross.columns:
            if cand.lower() in c.lower():
                return c
    return None

COL_ERR = find_col(["Human_Error_Probability_%","Human Error Probability %","human_error_probability","error_probability"])
COL_RISK = find_col(["Confusion Risk Score","Confusion_Risk_Score","confusion_risk_score"])
COL_MIS = find_col(["Most_Likely_Misclassification","Most Likely Misclassification","most_likely_misclassification"])

if COL_ERR is None or COL_RISK is None or COL_MIS is None:
    print("⚠️ Crosswalk column mapping:")
    print(" - role:", CROSS_ROLE_COL)
    print(" - error%:", COL_ERR)
    print(" - risk:", COL_RISK)
    print(" - likely misclass:", COL_MIS)
    raise KeyError("Crosswalk is missing one or more required columns (error%, risk score, likely misclassification).")

df_cross["major_role_group"] = df_cross[CROSS_ROLE_COL].map(norm)
df_cross["human_error_probability"] = pd.to_numeric(df_cross[COL_ERR], errors="coerce")
df_cross["confusion_risk_score"] = pd.to_numeric(df_cross[COL_RISK], errors="coerce")
df_cross["most_likely_misclassification"] = df_cross[COL_MIS].map(norm)

priors = df_cross[["major_role_group","human_error_probability","confusion_risk_score","most_likely_misclassification"]].dropna(subset=["major_role_group"])
priors = priors.drop_duplicates("major_role_group", keep="first")

print("Priors rows:", priors.shape[0])
display(priors.head(10))


In [ ]:
# ==== 5) Build record-level ambiguity signals from alternative_roles_analysis ====# The alternative_roles_analysis.csv contains ONE row per ROLE (not per job title)# We merge by major_role_group to get the alternative roles for each classified roledef to_list(x):    if x is None or (isinstance(x, float) and np.isnan(x)):        return []    if isinstance(x, list):        return [norm(v) for v in x if norm(v)]    s = str(x).strip()    if not s:        return []    # split by commas/semicolons/slashes/newlines    parts = re.split(r"[;,/\n]+", s)    return [norm(p) for p in parts if norm(p)]# Detect the "Classified Role" column in df_altalt_role_col = Nonefor c in df_alt.columns:    cl = c.lower().replace(" ", "").replace("_", "")    if cl in ["classifiedrole", "role", "majorrolegroup", "majorgroup"]:        alt_role_col = c        breakif alt_role_col is None:    raise KeyError(f"Could not find 'Classified Role' column in alternative_roles_analysis. Columns: {list(df_alt.columns)}")# Detect alt roles list columnalt_list_col = Nonefor c in df_alt.columns:    cl = c.lower()    if "other plausible" in cl or "other_plausible" in cl or "alternative" in cl or "alternatives" in cl or "other roles" in cl or "other_roles" in cl:        alt_list_col = c        breakif alt_list_col is None:    raise KeyError(f"Could not detect an alternate-roles list column in alternative_roles_analysis. Columns: {list(df_alt.columns)}")# Parse the alternative roles listdf_alt["alt_roles_list"] = df_alt[alt_list_col].apply(to_list)df_alt["alt_count"] = df_alt["alt_roles_list"].apply(len)# Normalize the role column for mergingdf_alt["major_role_group"] = df_alt[alt_role_col].map(norm)# Create a lookup for alternative roles by major_role_groupalt_lookup = df_alt[["major_role_group", "alt_roles_list", "alt_count"]].copy()alt_lookup = alt_lookup.drop_duplicates("major_role_group", keep="first")print("✅ Alternative roles by major_role_group:")print(f"   Found {len(alt_lookup)} role groups with alternatives")display(alt_lookup.head(10))

In [ ]:
# ==== 6) Critical confusion patterns (from universal prompt) ====
# Since we are not using full job description text, pattern detection is based on *role pairs* that are known to be high-risk.

CRITICAL_PAIRS = {
    ("Analyst","Auditor"),
    ("Auditor","Analyst"),
    ("Manager","Director"),
    ("Director","Manager"),
    ("Technician","Operator"),
    ("Operator","Technician"),
    ("Technician","Mechanic"),
    ("Mechanic","Technician"),
    ("Technician","Skilled Laborer"),
    ("Skilled Laborer","Technician"),
    ("Coordinator","Manager"),
    ("Manager","Coordinator"),
}

def pattern_hit(pred_role, likely_mis):
    a, b = norm(pred_role), norm(likely_mis)
    if not a or not b:
        return 0
    return 1 if (a,b) in CRITICAL_PAIRS else 0

print("Critical pairs loaded:", len(CRITICAL_PAIRS))


In [ ]:
# ==== 7) Compute Likelihood of Error Score (0–5) ====# Prepare priors from crosswalk# Normalize crosswalk role columnrole_key_col = Nonefor c in df_cross.columns:    if c.lower() in ["role","major_role_group","major role group","classified role"]:        role_key_col = c        breakif role_key_col is None:    raise KeyError(f"Could not find role key column in crosswalk. Columns: {list(df_cross.columns)}")df_cross["_role_key"] = df_cross[role_key_col].map(norm)def pick_col(possible_names):    for name in possible_names:        for c in df_cross.columns:            if c.lower() == name.lower():                return c    return Nonecol_err = pick_col(["Human_Error_Probability_%","Human Error Probability %","human_error_probability"])col_score = pick_col(["Confusion Risk Score","confusion_risk_score"])col_most = pick_col(["Most_Likely_Misclassification","most_likely_misclassification"])col_topmatch = pick_col(["Top Match Role","top_match_role"])priors = pd.DataFrame({    "major_role_group": df_cross["_role_key"],    "human_error_probability": pd.to_numeric(df_cross[col_err], errors="coerce") if col_err else np.nan,    "confusion_risk_score": pd.to_numeric(df_cross[col_score], errors="coerce") if col_score else np.nan,    "most_likely_misclassification": df_cross[col_most].map(norm) if col_most else "",    "top_match_role": df_cross[col_topmatch].map(norm) if col_topmatch else ""}).drop_duplicates(subset=["major_role_group"])# Merge job results + priors + alt signals BY ROLE (not by title)df = df_jobs.merge(priors, on="major_role_group", how="left")df = df.merge(alt_lookup, on="major_role_group", how="left")# Defaults if crosswalk missing for a roledf["human_error_probability"] = df["human_error_probability"].fillna(25.0)  # conservative defaultdf["confusion_risk_score"] = df["confusion_risk_score"].fillna(1.0)df["most_likely_misclassification"] = df["most_likely_misclassification"].fillna("")df["top_match_role"] = df["top_match_role"].fillna("")df["alt_roles_list"] = df["alt_roles_list"].apply(lambda x: x if isinstance(x, list) else [])df["alt_count"] = df["alt_count"].fillna(0).astype(int)# Pattern hit uses predicted role + most-likely misclassification from crosswalkdf["pattern_hit"] = df.apply(lambda r: pattern_hit(r["major_role_group"], r["most_likely_misclassification"]), axis=1)# Crosswalk confirmation: does crosswalk's top-match role show up among this record's plausible alternatives?def confirm_crosswalk(row):    tm = norm(row.get("top_match_role",""))    if not tm:        return 0    alts = row.get("alt_roles_list", [])    alts_norm = [norm(a) for a in alts]    return 1 if tm in alts_norm else 0df["crosswalk_confirmed"] = df.apply(confirm_crosswalk, axis=1)# Normalize components to 0..1P = (df["human_error_probability"] / 100).clip(0,1)C = (df["confusion_risk_score"] / 2).clip(0,1)     # assuming 0..2A = (df["alt_count"] / 4).clip(0,1)                # cap at 4 alternativesR = df["pattern_hit"].clip(0,1)M = df["crosswalk_confirmed"].clip(0,1)            # confirmation signal# Weighted error probability -> score (adds confirmation term)df["p_error"] = (0.40*P + 0.18*C + 0.22*A + 0.08*R + 0.12*M).clip(0,1)df["likelihood_error_score_0_5"] = (5 * df["p_error"]).round(1)# Helpful labelingdf["likelihood_band"] = pd.cut(    df["likelihood_error_score_0_5"],    bins=[-0.001, 1, 2, 3, 4, 5.001],    labels=["Very Low","Low","Moderate","High","Very High"])cols_out = [    "job_title_key",    "major_role_group",    "likelihood_error_score_0_5",    "likelihood_band",    "human_error_probability",    "confusion_risk_score",    "most_likely_misclassification",    "top_match_role",    "crosswalk_confirmed",    "alt_count",    "alt_roles_list",    "pattern_hit"]display(df[cols_out].head(10))print("✅ Scored records:", len(df))

In [ ]:
## Phase 2: Optional LLM Enhancement (Hybrid Approach)**The cells below implement the OPTIONAL Phase 2 of the hybrid approach.**✅ **Phase 1 is complete** - You now have deterministic likelihood scores for all records (cells 1-9).📊 **Hybrid strategy (Moderate+ only):**1. LLM will ONLY review records with risk band = **Moderate, High, or Very High**2. Low and Very Low risk records are skipped (deterministic scoring is highly accurate for these)3. This targets LLM evaluation where it adds most value**Risk Band Thresholds:**- **Very Low (0-1.0)**: No LLM review needed- **Low (1.0-2.0)**: No LLM review needed- **Moderate (2.0-3.0)**: ✅ LLM reviews these- **High (3.0-4.0)**: ✅ LLM reviews these- **Very High (4.0-5.0)**: ✅ LLM reviews these**Expected volume:** Typically 10-20% of records are Moderate or higher**Benefits:**- **Highly targeted**: Only reviews truly ambiguous classifications- **Cost-effective**: Minimal API usage (typically <20% of records)- **Efficient**: Fast for clear-cut cases, thorough for borderline cases**What the LLM evaluates:**- **(a)** The predicted role classification- **(b)** Crosswalk confusion priors- **(c)** Alternative role analysis outputs- **(d)** Classification justification text (if available)**Note:** The LLM does NOT see full job descriptions - only the classification metadata.---### Configuration: Choose your model below

## Phase 2: Optional LLM Enhancement (Hybrid Approach)**The cells below implement the OPTIONAL Phase 2 of the hybrid approach.**✅ **Phase 1 is complete** - You now have deterministic likelihood scores for all records (cells 1-9).📊 **Recommended hybrid strategy:**1. Filter records with likelihood_error_score > 2.0 (Moderate-High risk)2. Run LLM evaluation on this subset only (~20-30% of records typically)3. Use LLM insights to validate or adjust scores for borderline cases**Benefits:**- **Cost-effective**: Only use LLM API calls for high-risk records- **Targeted**: Focus AI evaluation where it adds most value- **Fast**: Deterministic scoring handles bulk of records instantly**What the LLM evaluates:**- **(a)** The predicted role classification- **(b)** Crosswalk confusion priors- **(c)** Alternative role analysis outputs- **(d)** Classification justification text (if available)**Note:** The LLM does NOT see full job descriptions - only the classification metadata.---### Configuration: Choose your model below

In [ ]:
# ==== 10) LLM judge with hybrid filtering ====import jsonimport reimport pandas as pdJUDGE_SYSTEM = """You are an auditing assistant for an HR job classification evaluation pipeline.You will be given a JSON record that includes:- job_title_original- major_role_group (the chosen classification)- grouping_justification (may be empty)- crosswalk signals (overall risk, top-match)- alt-role evidence (list of plausible alternatives)- likelihood_error_score_0_5 (deterministic score)Your job:1) Detect whether the justification is weak, hedge-heavy, or title-only.2) Detect whether the justification text strongly suggests the competing role (if provided).3) Assess if the likelihood_error_score seems appropriate given the evidence.4) Output ONLY valid JSON matching the schema below.Schema:{  "hedging_language": true/false,  "title_only_reasoning": true/false,  "mentions_competing_role_terms": true/false,  "score_assessment": "appropriate" | "too_low" | "too_high",  "confidence": "high" | "medium" | "low",  "notes": "short explanation (<=200 chars)"}"""HEDGE_PAT = re.compile(r"\b(aligns most closely|appears|seems|while|although|likely|generally)\b", re.I)def judge_record(row: dict, competing_terms=None):    just = (row.get("grouping_justification") or "").strip()    hedging = bool(HEDGE_PAT.search(just)) if just else False    # naive "title-only": justification mentions the title or role but lacks function words    title_only = False    if just:        if re.search(r"\b(title says|because the title|job title)\b", just, re.I):            title_only = True    mentions_competing = False    if just and competing_terms:        for t in competing_terms:            if t and re.search(r"\b" + re.escape(t) + r"\b", just, re.I):                mentions_competing = True                break    payload = {        "job_title_original": row.get("job_title_original"),        "major_role_group": row.get("major_role_group"),        "likelihood_error_score_0_5": row.get("likelihood_error_score_0_5"),        "likelihood_band": row.get("likelihood_band"),        "grouping_justification": just[:1200],        "crosswalk_overall_risk": row.get("confusion_risk_score"),        "crosswalk_top_match_role": row.get("top_match_role"),        "alt_roles": row.get("alt_roles_list", []),        "competing_terms": competing_terms or [],    }    user_prompt = json.dumps(payload, ensure_ascii=False)    txt = generate_text(JUDGE_SYSTEM, user_prompt, max_new_tokens=300, temperature=0.0)    # Best-effort JSON extraction    m = re.search(r"\{.*\}", txt, re.S)    if not m:        return {            "hedging_language": hedging,             "title_only_reasoning": title_only,            "mentions_competing_role_terms": mentions_competing,             "score_assessment": "unknown",            "confidence": "low",            "notes": "Model output not JSON; used heuristics."        }    try:        out = json.loads(m.group(0))    except Exception:        return {            "hedging_language": hedging,             "title_only_reasoning": title_only,            "mentions_competing_role_terms": mentions_competing,             "score_assessment": "unknown",            "confidence": "low",            "notes": "JSON parse failed; used heuristics."        }    # Ensure fields exist    out.setdefault("hedging_language", hedging)    out.setdefault("title_only_reasoning", title_only)    out.setdefault("mentions_competing_role_terms", mentions_competing)    out.setdefault("score_assessment", "unknown")    out.setdefault("confidence", "medium")    out.setdefault("notes", "")    return out# ========== HYBRID APPROACH: Filter for Moderate risk or higher ==========print("="*70)print("HYBRID APPROACH: LLM Evaluation")print("="*70)# Filter for Moderate, High, and Very High risk bands onlyRISK_BANDS_FOR_REVIEW = ["Moderate", "High", "Very High"]# Filter recordsmoderate_or_higher = df[df["likelihood_band"].isin(RISK_BANDS_FOR_REVIEW)].copy()print(f"\nTotal records: {len(df)}")print(f"\nRisk band distribution:")print(df["likelihood_band"].value_counts().sort_index())print(f"\n{'='*70}")print(f"Records selected for LLM review: {len(moderate_or_higher)}")print(f"Percentage for LLM review: {100*len(moderate_or_higher)/len(df):.1f}%")print(f"Risk bands included: {', '.join(RISK_BANDS_FOR_REVIEW)}")if len(moderate_or_higher) == 0:    print("\n✅ No Moderate or higher risk records found. Skipping LLM evaluation.")    print("   All records are Very Low or Low risk - deterministic scoring is sufficient.")else:    print(f"\n🔍 Running LLM evaluation on {len(moderate_or_higher)} Moderate+ risk records...")    print("This may take a few minutes depending on your model/API...")        # Breakdown by band    print("\nBreakdown of records for LLM review:")    print(moderate_or_higher["likelihood_band"].value_counts().sort_index())        # Optionally load model if using LOCAL_TRANSFORMERS    # load_local_model(MODEL_ID, use_4bit=True)        # Run judge on moderate+ risk subset    judged = []    for idx, (_, r) in enumerate(moderate_or_higher.iterrows(), 1):        if idx % 10 == 0:            print(f"  Progress: {idx}/{len(moderate_or_higher)}")                # Get competing terms from alt_roles_list        competing = r.get("alt_roles_list", [])[:3]  # Top 3 alternatives                result = judge_record(r.to_dict(), competing_terms=competing)        result["job_title_key"] = r["job_title_key"]        judged.append(result)        judged_df = pd.DataFrame(judged)        # Merge LLM judgments back to moderate+ records    moderate_enhanced = moderate_or_higher.merge(judged_df, on="job_title_key", how="left")        print("\n" + "="*70)    print("LLM Evaluation Results")    print("="*70)        print("\nConfidence distribution:")    if "confidence" in judged_df.columns:        print(judged_df["confidence"].value_counts())        print("\nScore assessment distribution:")    if "score_assessment" in judged_df.columns:        print(judged_df["score_assessment"].value_counts())        # Show examples by risk band    print("\n" + "="*70)    print("Sample Cases by Risk Band (with LLM Assessment)")    print("="*70)        display_cols = [        "job_title_key",        "major_role_group",         "likelihood_band",        "likelihood_error_score_0_5",        "confidence",        "score_assessment",        "hedging_language",        "notes"    ]        for band in ["Very High", "High", "Moderate"]:        band_records = moderate_enhanced[moderate_enhanced["likelihood_band"] == band]        if len(band_records) > 0:            print(f"\n--- {band} Risk Records ({len(band_records)} total) ---")            display(band_records.sort_values("likelihood_error_score_0_5", ascending=False)[display_cols].head(5))        print("\n✅ LLM evaluation complete!")    print("\nNext steps:")    print("  - Review Very High risk records first")    print("  - Focus on records where confidence = 'low' or score_assessment = 'too_low'")    print("  - Export results with LLM insights (next cell)")    print(f"\nLow/Very Low risk records ({len(df) - len(moderate_or_higher)}) were not reviewed by LLM")    print("  (deterministic scoring is highly accurate for clear-cut cases)")

In [ ]:
# ==== 10) LLM judge with hybrid filtering ====import jsonimport reimport pandas as pdJUDGE_SYSTEM = """You are an auditing assistant for an HR job classification evaluation pipeline.You will be given a JSON record that includes:- job_title_original- major_role_group (the chosen classification)- grouping_justification (may be empty)- crosswalk signals (overall risk, top-match)- alt-role evidence (list of plausible alternatives)- likelihood_error_score_0_5 (deterministic score)Your job:1) Detect whether the justification is weak, hedge-heavy, or title-only.2) Detect whether the justification text strongly suggests the competing role (if provided).3) Assess if the likelihood_error_score seems appropriate given the evidence.4) Output ONLY valid JSON matching the schema below.Schema:{  "hedging_language": true/false,  "title_only_reasoning": true/false,  "mentions_competing_role_terms": true/false,  "score_assessment": "appropriate" | "too_low" | "too_high",  "confidence": "high" | "medium" | "low",  "notes": "short explanation (<=200 chars)"}"""HEDGE_PAT = re.compile(r"\b(aligns most closely|appears|seems|while|although|likely|generally)\b", re.I)def judge_record(row: dict, competing_terms=None):    just = (row.get("grouping_justification") or "").strip()    hedging = bool(HEDGE_PAT.search(just)) if just else False    # naive "title-only": justification mentions the title or role but lacks function words    title_only = False    if just:        if re.search(r"\b(title says|because the title|job title)\b", just, re.I):            title_only = True    mentions_competing = False    if just and competing_terms:        for t in competing_terms:            if t and re.search(r"\b" + re.escape(t) + r"\b", just, re.I):                mentions_competing = True                break    payload = {        "job_title_original": row.get("job_title_original"),        "major_role_group": row.get("major_role_group"),        "likelihood_error_score_0_5": row.get("likelihood_error_score_0_5"),        "grouping_justification": just[:1200],        "crosswalk_overall_risk": row.get("confusion_risk_score"),        "crosswalk_top_match_role": row.get("top_match_role"),        "alt_roles": row.get("alt_roles_list", []),        "competing_terms": competing_terms or [],    }    user_prompt = json.dumps(payload, ensure_ascii=False)    txt = generate_text(JUDGE_SYSTEM, user_prompt, max_new_tokens=300, temperature=0.0)    # Best-effort JSON extraction    m = re.search(r"\{.*\}", txt, re.S)    if not m:        return {            "hedging_language": hedging,             "title_only_reasoning": title_only,            "mentions_competing_role_terms": mentions_competing,             "score_assessment": "unknown",            "confidence": "low",            "notes": "Model output not JSON; used heuristics."        }    try:        out = json.loads(m.group(0))    except Exception:        return {            "hedging_language": hedging,             "title_only_reasoning": title_only,            "mentions_competing_role_terms": mentions_competing,             "score_assessment": "unknown",            "confidence": "low",            "notes": "JSON parse failed; used heuristics."        }    # Ensure fields exist    out.setdefault("hedging_language", hedging)    out.setdefault("title_only_reasoning", title_only)    out.setdefault("mentions_competing_role_terms", mentions_competing)    out.setdefault("score_assessment", "unknown")    out.setdefault("confidence", "medium")    out.setdefault("notes", "")    return out# ========== HYBRID APPROACH: Filter for high-risk records ==========print("="*70)print("HYBRID APPROACH: LLM Evaluation")print("="*70)# Define risk thresholdRISK_THRESHOLD = 2.0  # Only evaluate scores >= 2.0# Filter high-risk recordshigh_risk = df[df["likelihood_error_score_0_5"] >= RISK_THRESHOLD].copy()print(f"\nTotal records: {len(df)}")print(f"High-risk records (score >= {RISK_THRESHOLD}): {len(high_risk)}")print(f"Percentage for LLM review: {100*len(high_risk)/len(df):.1f}%")if len(high_risk) == 0:    print("\n✅ No high-risk records found. Skipping LLM evaluation.")else:    print(f"\n🔍 Running LLM evaluation on {len(high_risk)} records...")    print("This may take a few minutes depending on your model/API...")        # Optionally load model if using LOCAL_TRANSFORMERS    # load_local_model(MODEL_ID, use_4bit=True)        # Run judge on high-risk subset    judged = []    for idx, (_, r) in enumerate(high_risk.iterrows(), 1):        if idx % 10 == 0:            print(f"  Progress: {idx}/{len(high_risk)}")                # Get competing terms from alt_roles_list        competing = r.get("alt_roles_list", [])[:3]  # Top 3 alternatives                result = judge_record(r.to_dict(), competing_terms=competing)        result["job_title_key"] = r["job_title_key"]        judged.append(result)        judged_df = pd.DataFrame(judged)        # Merge LLM judgments back to high_risk records    high_risk_enhanced = high_risk.merge(judged_df, on="job_title_key", how="left")        print("\n" + "="*70)    print("LLM Evaluation Results")    print("="*70)        print("\nConfidence distribution:")    if "confidence" in judged_df.columns:        print(judged_df["confidence"].value_counts())        print("\nScore assessment distribution:")    if "score_assessment" in judged_df.columns:        print(judged_df["score_assessment"].value_counts())        # Show examples of interesting cases    print("\n" + "="*70)    print("Sample High-Risk Cases with LLM Assessment")    print("="*70)        display_cols = [        "job_title_key",        "major_role_group",         "likelihood_error_score_0_5",        "confidence",        "score_assessment",        "hedging_language",        "notes"    ]        # Show top 10 by score    if len(high_risk_enhanced) > 0:        display(high_risk_enhanced.sort_values("likelihood_error_score_0_5", ascending=False)[display_cols].head(10))        print("\n✅ LLM evaluation complete!")    print("\nNext steps:")    print("  - Review cases where score_assessment = 'too_low' or 'too_high'")    print("  - Focus on records with confidence = 'low'")    print("  - Export results with LLM insights (next cell)")

In [ ]:
# ==== 11) Export results (with optional LLM enhancements) ====import os, jsonOUTDIR = "/content/output_likelihood_error"os.makedirs(OUTDIR, exist_ok=True)# Determine if we have LLM judgmentshas_llm_results = 'judged_df' in locals() and len(judged_df) > 0if has_llm_results:    print("="*70)    print("EXPORTING HYBRID RESULTS (Deterministic + LLM)")    print("="*70)        # Merge LLM judgments back into main dataframe    llm_cols = ["job_title_key", "hedging_language", "title_only_reasoning",                 "mentions_competing_role_terms", "score_assessment", "confidence", "notes"]        # Keep existing columns from judged_df that exist    available_llm_cols = [c for c in llm_cols if c in judged_df.columns]        df_export = df.merge(judged_df[available_llm_cols], on="job_title_key", how="left")        # Add flag for LLM reviewed    df_export["llm_reviewed"] = df_export["confidence"].notna()        print(f"\nRecords with LLM review: {df_export['llm_reviewed'].sum()}")    print(f"Records deterministic only: {(~df_export['llm_reviewed']).sum()}")    else:    print("="*70)    print("EXPORTING DETERMINISTIC RESULTS ONLY")    print("="*70)    print("\nNo LLM evaluation was run. Exporting deterministic scores only.")    df_export = df.copy()    df_export["llm_reviewed"] = False# Define output columnsbase_cols = [    "job_title_key",    "major_role_group",    "likelihood_error_score_0_5",    "likelihood_band",    "human_error_probability",    "confusion_risk_score",    "most_likely_misclassification",    "top_match_role",    "crosswalk_confirmed",    "alt_count",    "alt_roles_list",    "pattern_hit"]# Add LLM columns if availableif has_llm_results:    llm_export_cols = ["llm_reviewed", "confidence", "score_assessment",                        "hedging_language", "title_only_reasoning",                        "mentions_competing_role_terms", "notes"]    export_cols = base_cols + [c for c in llm_export_cols if c in df_export.columns]else:    export_cols = base_cols + ["llm_reviewed"]# Export CSVcsv_path = os.path.join(OUTDIR, "Job_Classifications_Batch_with_Likelihood_Error.csv")df_export[export_cols].to_csv(csv_path, index=False)# Export JSON with full recordsjson_path = os.path.join(OUTDIR, "Job_Classifications_Batch_with_Likelihood_Error.json")# Merge scores back to original job recordsrecords = df_jobs.copy()records["job_title_key"] = records[TITLE_COL].map(norm)# Create score mapsscore_map = df_export.set_index("job_title_key")["likelihood_error_score_0_5"].to_dict()band_map = df_export.set_index("job_title_key")["likelihood_band"].astype(str).to_dict()records["likelihood_error_score_0_5"] = records["job_title_key"].map(score_map)records["likelihood_band"] = records["job_title_key"].map(band_map)# Add LLM fields if availableif has_llm_results:    for col in ["confidence", "score_assessment", "llm_reviewed"]:        if col in df_export.columns:            col_map = df_export.set_index("job_title_key")[col].to_dict()            records[col] = records["job_title_key"].map(col_map)with open(json_path, "w", encoding="utf-8") as f:    json.dump(records.drop(columns=["job_title_key"]).to_dict(orient="records"), f, ensure_ascii=False, indent=2)print("\n✅ Exported files:")print(f"   - {csv_path}")print(f"   - {json_path}")# Generate summary reportprint("\n" + "="*70)print("SUMMARY REPORT")print("="*70)print(f"\nTotal records processed: {len(df_export)}")print(f"\nLikelihood Score Distribution:")print(df_export["likelihood_band"].value_counts().sort_index())if has_llm_results:    print(f"\nLLM-Enhanced Records: {df_export['llm_reviewed'].sum()}")    print(f"\nLLM Confidence Distribution (for reviewed records):")    reviewed = df_export[df_export["llm_reviewed"]]    if len(reviewed) > 0 and "confidence" in reviewed.columns:        print(reviewed["confidence"].value_counts())        print(f"\nLLM Score Assessment (for reviewed records):")    if "score_assessment" in reviewed.columns:        print(reviewed["score_assessment"].value_counts())        # Flag concerning cases    concerning = reviewed[        (reviewed["score_assessment"] == "too_low") |         (reviewed["confidence"] == "low")    ]        if len(concerning) > 0:        print(f"\n⚠️ {len(concerning)} cases flagged for manual review:")        print("   (score_assessment='too_low' OR confidence='low')")print("\n" + "="*70)# Download filesfrom google.colab import filesfiles.download(csv_path)files.download(json_path)print("\n✅ Export complete! Files downloaded to your local machine.")